In [1]:
# =============================================================================
# STEP 4 - SELECTIVE PREDICTION ON THE CLASSES THAT ACTUALLY FAIL
#
# Figure 7 currently reports abstention curves for the three preregistered focal
# classes. Two of those, UGR'16 nerisbotnet and CIC-IoT-2023 Web, were already at
# or near nominal coverage, so the policy was being asked to repair something that
# was not broken. That is the weakest possible test of an abstention policy.
#
# The UGR'16 classes that DO fail are scan11 (coverage 0.535) and scan44 (0.799),
# and they are the interesting case for a second reason: scan11 has a misroute
# rate of 0.56, the same regime in which the label-free monitor goes blind. If
# abstention also fails there, the two blind spots have a common cause, which is
# a stronger claim than either alone.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib, time
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
from conformal import conformal_q
import numpy as np, pandas as pd
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY; R=getattr(config,'N_MATCHED_DRAWS',10)
def dseed(*p): return int(hashlib.sha256('|'.join(map(str,p)).encode()).hexdigest(),16)%(2**32)
print('ready | alpha', ALPHA, '| draws', R)


Mounted at /content/drive
ready | alpha 0.05 | draws 10


In [2]:
# =============================================================================
# Cell 2 - the policy, unchanged from Step-0 nb30 so the curves are comparable,
# but evaluated for ANY class rather than only the preregistered focal one.
#
#     m(x) = max over classes c of ( q_c - s(x, c) )
#
# Escalate in ascending margin. Classes whose quantile is infinite under the
# feasibility rule are excluded from the maximisation. The decision never
# consults a label; labels enter only when scoring the outcome.
# =============================================================================
def aps_scores(P, rng):
    o=np.argsort(-P,axis=1); sp=np.take_along_axis(P,o,1); cum=np.cumsum(sp,1)
    U=rng.random(len(P))[:,None]; ss=cum-(1-U)*sp
    out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out

FRACS=np.round(np.arange(0.0,0.51,0.025),3)

def selective_curve(S_ev, y_ev, q, class_indices, class_names):
    """Label-free abstention; per-class outcomes for every class in class_indices."""
    finite=np.isfinite(q)
    margin=np.where(finite[None,:], q[None,:]-S_ev, -np.inf).max(axis=1)
    order=np.argsort(margin, kind='stable')
    n=len(margin)
    covered=(S_ev[np.arange(n), y_ev] <= q[y_ev])
    rows=[]
    for f in FRACS:
        k=int(round(f*n))
        abst=np.zeros(n,bool); abst[order[:k]]=True; keep=~abst
        for ci,cn in zip(class_indices, class_names):
            m=y_ev==ci; km=keep&m
            rows.append({'class':cn,'escalated_frac':float(f),
                'coverage_retained':float(covered[km].mean()) if km.any() else np.nan,
                'retained_frac':float(km.sum()/m.sum()) if m.any() else np.nan,
                'baseline_coverage':float(covered[m].mean()) if m.any() else np.nan})
        rows.append({'class':'__marginal__','escalated_frac':float(f),
            'coverage_retained':float(covered[keep].mean()) if keep.any() else np.nan,
            'retained_frac':float(keep.sum()/n),'baseline_coverage':float(covered.mean())})
    return rows
print('policy ready | escalation grid', FRACS[0], '->', FRACS[-1])


policy ready | escalation grid 0.0 -> 0.5


In [3]:
# =============================================================================
# Cell 3 - UGR'16, all five classes, so the failing scan classes are measured
# alongside the ones that hold and the contrast is visible in one place.
# =============================================================================
UGR=config.DATASETS_DIR/'ugr16'
us=pd.read_parquet(UGR/'july_week5.parquet'); ut=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (us,ut): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
us=us[us.label.isin(UK)].reset_index(drop=True); ut=ut[ut.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}; K=len(UCL)
def strat(df,fr,seed,col='label'):
    rg=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(ff))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rg.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)]+=n-c.sum(); kk=0
        for a2,q in zip(nm,c): a.loc[idx[kk:kk+q]]=a2; kk+=q
    return a
us=us.assign(partition=strat(us,config.SPLIT_FRACTIONS,20260725).values)
y_sp=us[us.partition=='source_cal_pool']['label'].map(U2I).to_numpy()
y_tg=ut['label'].map(U2I).to_numpy()
mm=min(len(y_sp), len(y_tg)//2)
print('UGR source pool', len(y_sp), '| target', len(y_tg), '| matched draw size', mm)

rows=[]; t0=time.time()
for f in sorted((config.DATA_DIR/'ugr16_probs').glob('ugr16__*.npz')):
    _,arch,sd=Path(f).stem.split('__'); seed=int(sd.replace('seed',''))
    d=np.load(f); Psp,Ptg=d['srcpool'],d['target']
    for draw in range(R):
        rng=np.random.default_rng(dseed('ugr16',seed,arch,draw))
        tp=rng.permutation(len(y_tg)); de=tp[:mm]; sc=rng.permutation(len(y_sp))[:mm]
        S_ev=aps_scores(Ptg[de], np.random.default_rng(dseed('ugr16',seed,arch,draw,'e')))
        S_sc=aps_scores(Psp[sc], np.random.default_rng(dseed('ugr16',seed,arch,draw,'s')))
        ysc=y_sp[sc]; tc=S_sc[np.arange(len(ysc)),ysc]
        q=np.array([conformal_q(tc[ysc==k],ALPHA)[0] for k in range(K)])
        for r in selective_curve(S_ev, y_tg[de], q, list(range(K)), UCL):
            r.update({'dataset':'ugr16','arch':arch,'seed':seed,'draw':draw}); rows.append(r)
sel=pd.DataFrame(rows)
print(f'rows {len(sel):,} | {time.time()-t0:.0f}s')


UGR source pool 60000 | target 400000 | matched draw size 60000
rows 37,800 | 79s


In [4]:
# =============================================================================
# Cell 4 - does abstention rescue the classes that actually fail?
# =============================================================================
g=sel.groupby(['class','escalated_frac'])[['coverage_retained','retained_frac']].mean().reset_index()
NOM=1-ALPHA
print(f'COVERAGE ON RETAINED ALERTS, by escalation rate (nominal {NOM})')
piv=g.pivot(index='escalated_frac', columns='class', values='coverage_retained')
show=[c for c in ['scan11','scan44','nerisbotnet','dos','background','__marginal__'] if c in piv.columns]
print(piv[show].loc[[0.0,0.10,0.20,0.30,0.40,0.50]].round(4).to_string())

print(f'\nFRACTION OF EACH CLASS STILL RETAINED (guard against deleting the class)')
piv2=g.pivot(index='escalated_frac', columns='class', values='retained_frac')
print(piv2[show].loc[[0.0,0.10,0.20,0.30,0.40,0.50]].round(4).to_string())

GUARD=0.20
print('\nVERDICT per class:')
head=[]
for c in [x for x in show if x!='__marginal__']:
    gg=g[g['class']==c].sort_values('escalated_frac')
    base=float(gg.iloc[0].coverage_retained)
    ok=gg[(gg.coverage_retained>=NOM)&(gg.retained_frac>=GUARD)]
    best=gg.loc[gg.coverage_retained.idxmax()]
    reach=float(ok.iloc[0].escalated_frac) if len(ok) else None
    head.append({'class':c,'baseline':round(base,4),
                 'escalation_to_nominal':reach,
                 'best_coverage':round(float(best.coverage_retained),4),
                 'at_escalation':round(float(best.escalated_frac),3),
                 'retained_at_best':round(float(best.retained_frac),3),
                 'gain_at_50pct':round(float(gg.iloc[-1].coverage_retained)-base,4)})
    tag = f'reaches nominal at {reach:.0%}' if reach is not None else 'never reaches nominal within a 50% budget'
    print(f'  {c:12s} baseline {base:.4f} -> best {float(best.coverage_retained):.4f} '
          f'at {float(best.escalated_frac):.0%} escalation | {tag}')
H=pd.DataFrame(head)

print('\nREAD:')
print('  scan11 and scan44 are the UGR classes that fail. If abstention leaves them')
print('  essentially unchanged while lifting the marginal, the same identifiability limit')
print('  that blinds the monitor also defeats triage: the alerts that break class-conditional')
print('  coverage are confident misroutes, which carry LARGE margins and are escalated last.')


COVERAGE ON RETAINED ALERTS, by escalation rate (nominal 0.95)
class           scan11  scan44  nerisbotnet     dos  background  __marginal__
escalated_frac                                                               
0.0             0.5349  0.7989       0.9471  0.9502      0.9491        0.8785
0.1             0.5514  0.8269       0.9773  1.0000      0.9955        0.9160
0.2             0.5594  0.8331       0.9813  1.0000      0.9954        0.9172
0.3             0.5735  0.8422       0.9914  1.0000      0.9954        0.9209
0.4             0.5897  0.8500       0.9917  1.0000      0.9956        0.9244
0.5             0.5978  0.8527       0.9921  1.0000      0.9958        0.9260

FRACTION OF EACH CLASS STILL RETAINED (guard against deleting the class)
class           scan11  scan44  nerisbotnet     dos  background  __marginal__
escalated_frac                                                               
0.0             1.0000  1.0000       1.0000  1.0000      1.0000           1.0
0.1  

In [ ]:
# =============================================================================
# Cell 5 - figure, save, commit. Replaces Figure 7 with the failing classes shown.
# =============================================================================
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.spines.top':False,
                     'axes.spines.right':False,'figure.dpi':300})
COL={'scan11':'#b3261e','scan44':'#c65911','nerisbotnet':'#4a5a2f',
     'dos':'#2f4b7c','background':'#777777','__marginal__':'#000000'}
LBL={'__marginal__':'marginal (all classes)'}
fig,axs=plt.subplots(1,2,figsize=(10.0,4.3))
for c in show:
    gg=g[g['class']==c].sort_values('escalated_frac')
    axs[0].plot(100*gg.escalated_frac, gg.coverage_retained, marker='o', ms=3.5,
                color=COL.get(c,'#999'), ls='--' if c=='__marginal__' else '-',
                label=LBL.get(c,c))
    if c!='__marginal__':
        axs[1].plot(100*gg.escalated_frac, 100*gg.retained_frac, marker='o', ms=3.5,
                    color=COL.get(c,'#999'), label=c)
axs[0].axhline(NOM, color='#666', ls=':', lw=1, label='nominal 0.95')
axs[0].set_xlabel('alerts escalated to an analyst (%)'); axs[0].set_ylabel('coverage on retained alerts')
axs[0].set_title('(a) UGR\u201916: abstention and the classes that fail')
axs[0].legend(fontsize=7.5, frameon=False, loc='lower right')
axs[1].axhline(100*GUARD, color='#b3261e', ls=':', lw=1, label='retention guard 20%')
axs[1].set_xlabel('alerts escalated to an analyst (%)'); axs[1].set_ylabel('class alerts retained (%)')
axs[1].set_title('(b) how much of each class survives'); axs[1].legend(fontsize=7.5, frameon=False)
fig.tight_layout(); fig.savefig(RD/'selective_prediction_ugr_classes.png',bbox_inches='tight',facecolor='white')
print('figure saved: selective_prediction_ugr_classes.png')

sel.to_csv(RD/'selective_prediction_ugr_all_classes.csv', index=False)
g.to_csv(RD/'selective_prediction_ugr_curve.csv', index=False)
H.to_csv(RD/'selective_prediction_ugr_headline.csv', index=False)
(RD/'selective_prediction_ugr_verdict.json').write_text(json.dumps({
 'why':'Figure 7 previously showed only preregistered focal classes, two of which were '
       'already at nominal; these are the UGR classes that actually fail',
 'policy':'m(x)=max_c(q_c - s(x,c)) over classes with a finite quantile; escalate ascending; '
          'the decision never uses a label',
 'nominal':NOM,'retention_guard':GUARD,'headline':H.to_dict('records')}, indent=2, default=str))
print('saved curve, headline and verdict')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','step 4: selective prediction on the UGR classes that actually fail (scan11, scan44)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


figure saved: selective_prediction_ugr_classes.png
saved curve, headline and verdict
